### **Source Code Analisis Pola Asosiasi Produk pada Data Transaksi Ritel Menggunakan Algoritma Apriori**

In [ ]:
# =====================================
# KONFIGURASI (BIAR OUTPUT BERSIH)
# =====================================
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# =====================================
# INSTALL LIBRARY
# =====================================
#!pip install mlxtend plotly

In [ ]:
# =====================================
# IMPORT LIBRARY
# =====================================
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import plotly.express as px
import networkx as nx
import matplotlib.pyplot as plt
import ast

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# LOAD DATASET
# =====================================
from google.colab import drive
drive.mount('/content/drive')

Retail = pd.read_csv(
    "/content/drive/MyDrive/Project Prak Datmin/Retail.csv",
    encoding="latin1",
    low_memory=False
)

print(Retail.head())
print("Jumlah data awal:", Retail.shape[0])

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# SAMPLING DATA (1000 TRANSAKSI)
# =====================================
Retail = Retail.sample(n=1000, random_state=42)
print("Jumlah data setelah sampling:", Retail.shape[0])

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Menampilkan 5 data awal hasil sampling
print("\n5 Data Awal Setelah Sampling:")
display(Retail.head())

# Menampilkan 5 data akhir hasil sampling
print("\n5 Data Akhir Setelah Sampling:")
display(Retail.tail())

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# STATISTIK DESKRIPTIF
# =====================================
Retail_sample = Retail.sample(n=1000, random_state=42).reset_index(drop=True)

print("Jumlah data sampling:", Retail_sample.shape[0])

stat_desc = Retail_sample[['Total_Items', 'Total_Cost']].describe()
display(stat_desc)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# Distribusi Jumlah Item per Transaksi
# =====================================

# Hitung jumlah transaksi untuk setiap Total_Items
items_count = Retail['Total_Items'].value_counts().sort_index()

# Pastikan semua nilai 1–10 muncul
items_count = items_count.reindex(range(1, 11), fill_value=0).reset_index()
items_count.columns = ['Total_Items', 'Jumlah_Transaksi']

fig = px.bar(
    items_count,
    x='Total_Items',
    y='Jumlah_Transaksi',
    title='Distribusi Jumlah Item per Transaksi',
    text='Jumlah_Transaksi'
)


fig.update_traces(
    marker_color='#4C78A8',
    textposition='outside'
)

fig.update_layout(
    xaxis_title='Jumlah Item per Transaksi',
    yaxis_title='Jumlah Transaksi',
    bargap=0.35,
    uniformtext_minsize=10,
    uniformtext_mode='hide'
)

fig.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# Distribusi Metode Pembayaran (Pie Chart)
# =====================================
payment_df = Retail['Payment_Method'].value_counts().reset_index()
payment_df.columns = ['Payment_Method', 'Jumlah_Transaksi']

fig = px.pie(
    payment_df,
    names='Payment_Method',
    values='Jumlah_Transaksi',
    title='Distribusi Metode Pembayaran',
    color_discrete_sequence=[
        '#1f3b4d',
        '#4f6d7a',
        '#9db4c0',
        '#dbe9ee'
    ]
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    marker=dict(line=dict(color='white', width=1))
)

fig.update_layout(
    font=dict(size=12),
    showlegend=True
)

fig.show()

In [ ]:
# =====================================
# Distribusi Store Type
# =====================================
store_df = Retail['Store_Type'].value_counts().reset_index()
store_df.columns = ['Store_Type', 'Jumlah_Transaksi']

fig = px.bar(
    store_df,
    x='Store_Type',
    y='Jumlah_Transaksi',
    title="Distribusi Transaksi Berdasarkan Store Type",
    text='Jumlah_Transaksi',
    color_discrete_sequence=['#4C6A92']
)

# posisi & format angka
fig.update_traces(
    textposition='outside',
    texttemplate='%{text:,}'  # pakai pemisah ribuan
)

fig.update_layout(
    xaxis_title="Store Type",
    yaxis_title="Jumlah Transaksi",
    showlegend=False,
    bargap=0.35,
    plot_bgcolor='white'
)

fig.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

season_df = Retail['Season'].value_counts().reindex(
    ['Spring', 'Summer', 'Fall', 'Winter']
).reset_index()
season_df.columns = ['Season', 'Jumlah_Transaksi']

fig = px.bar(
    season_df,
    x='Season',
    y='Jumlah_Transaksi',
    text='Jumlah_Transaksi',
    color_discrete_sequence=['#4E6E81'],
    title='Distribusi Transaksi Berdasarkan Musim'
)

fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Musim',
    yaxis_title='Jumlah Transaksi',
    bargap=0.35,
    showlegend=False
)

fig.show()

## Preprocessing

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

#=====================================
#  Missing Value
# =====================================
display(Retail.isnull().sum())

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
#  Duplikasi Data
# =====================================
print("Jumlah data duplikat:", Retail.duplicated().sum())

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# Ubah kolom Product dari string ke list (Pembentukan Data Transaksi)
# =====================================
Retail['Product'] = Retail['Product'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [ ]:
# Bentuk list transaksi
transactions_list = Retail['Product'].tolist()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# TRANSFORMASI ONE-HOT ENCODING
# =====================================
te = TransactionEncoder()
te_ary = te.fit(transactions_list).transform(transactions_list)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

In [ ]:
# UBAH True/False → 1/0
df_encoded = df_encoded.astype(int)

print("5 baris pertama dari data hasil one-hot encoding:")
display(df_encoded.head())

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# APRIORI FREQUENT ITEMSET
# =====================================
frequent_itemsets = apriori(
    df_encoded,
    min_support=0.001,
    use_colnames=True,
    max_len=2
)

display(frequent_itemsets.sort_values(by='support', ascending=False).head(10))

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# Frequent 1-Itemsets
# =====================================

freq_1 = frequent_itemsets[
    frequent_itemsets['itemsets'].apply(lambda x: len(x) == 1)
].sort_values(by='support', ascending=False)

# Reset index & buat kolom No
freq_1 = freq_1.reset_index(drop=True)
freq_1.insert(0, 'No', range(1, len(freq_1) + 1))

# Ubah frozenset -> string
freq_1['Itemsets'] = freq_1['itemsets'].apply(lambda x: ', '.join(list(x)))

# Pilih kolom akhir
freq_1_display = freq_1[['No', 'support', 'Itemsets']]

display(freq_1_display.head(20))

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# Frequent 2-Itemsets
# =====================================

freq_2 = frequent_itemsets[
    frequent_itemsets['itemsets'].apply(lambda x: len(x) == 2)
].sort_values(by='support', ascending=False)

# Reset index & buat kolom No
freq_2 = freq_2.reset_index(drop=True)
freq_2.insert(0, 'No', range(1, len(freq_2) + 1))

# Ubah frozenset -> string
freq_2['Itemsets'] = freq_2['itemsets'].apply(lambda x: ', '.join(list(x)))

# Pilih kolom akhir
freq_2_display = freq_2[['No', 'support', 'Itemsets']]

display(freq_2_display.head(20))

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# ASSOCIATION RULES
# =====================================
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=0.3
)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# TAMPILKAN ATURAN ASOSIASI
# =====================================
rules_display = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
print("Aturan Asosiasi yang Ditemukan:")
print(rules_display.sort_values(by='lift', ascending=False).head(10))

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# AMBIL 50 ATURAN TERATAS
# =====================================
top_rules = rules.sort_values(by='lift', ascending=False).head(50).copy()

# Convert frozenset ke string (UNTUK PLOT)
top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ', '.join(list(x)))

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# =====================================
# Scatter Plot Aturan Asosiasi (FIX)
# =====================================

rules_plot = rules.copy()

# Ubah frozenset ke string
rules_plot['antecedents'] = rules_plot['antecedents'].apply(
    lambda x: ', '.join(list(x))
)
rules_plot['consequents'] = rules_plot['consequents'].apply(
    lambda x: ', '.join(list(x))
)

fig = px.scatter(
    rules_plot,
    x="support",
    y="confidence",
    size="lift",
    color="lift",
    color_continuous_scale="Plasma",
    hover_data=["antecedents", "consequents"],
    title="Scatter Plot Aturan Asosiasi"
)

fig.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

rules_graph = rules.sort_values(by='lift', ascending=False).head(15)

G = nx.DiGraph()

for _, row in rules_graph.iterrows():
    for a in row['antecedents']:
        for c in row['consequents']:
            G.add_edge(a, c, lift=row['lift'])

plt.figure(figsize=(12,9))
pos = nx.spring_layout(G, k=0.8, seed=42)

edges = G.edges(data=True)
edge_lift = [d['lift'] for (_, _, d) in edges]

# =====================
# NODE
# =====================
nx.draw_networkx_nodes(
    G, pos,
    node_size=1500,
    node_color="#8ecae6",
    edgecolors="black"
)

# =====================
# EDGE (SATU WARNA)
# =====================
nx.draw_networkx_edges(
    G, pos,
    width=[l * 1.8 for l in edge_lift],
    edge_color="#6c757d",
    alpha=0.9,
    connectionstyle="arc3,rad=0.15"
)

# =====================
# LABEL
# =====================
nx.draw_networkx_labels(G, pos, font_size=9)

plt.title("Network Graph Aturan Asosiasi Terkuat (Apriori) - Retail Dataset")
plt.axis("off")
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
# Ambil 10 aturan asosiasi dengan lift tertinggi
top_rules = rules.sort_values(by='lift', ascending=False).head(10)

# Pilih kolom penting
bundling_recommendation = top_rules[
    ['antecedents', 'consequents', 'support', 'confidence', 'lift']
].copy()

# Ubah frozenset ke string
bundling_recommendation['Produk 1'] = bundling_recommendation['antecedents'].apply(
    lambda x: ', '.join(list(x))
)
bundling_recommendation['Produk 2'] = bundling_recommendation['consequents'].apply(
    lambda x: ', '.join(list(x))
)

bundling_recommendation = bundling_recommendation[
    ['Produk 1', 'Produk 2', 'confidence', 'lift']
]

display(bundling_recommendation)